#ProteinMPNN
This notebook is intended as a quick demo, more features to come!

In [ ]:
#@title Clone github repo
import json, time, os, sys, glob

if not os.path.isdir("ProteinMPNN"):
  os.system("git clone -q https://github.com/dauparas/ProteinMPNN.git")
sys.path.append('/content/ProteinMPNN')

In [ ]:
#@title Setup Model
import matplotlib.pyplot as plt
import shutil
import warnings
import numpy as np
import torch
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split, Subset
import copy
import torch.nn as nn
import torch.nn.functional as F
import random
import os.path
from protein_mpnn_utils import loss_nll, loss_smoothed, gather_edges, gather_nodes, gather_nodes_t, cat_neighbors_nodes, _scores, _S_to_seq, tied_featurize, parse_PDB
from protein_mpnn_utils import StructureDataset, StructureDatasetPDB, ProteinMPNN

device = torch.device("cuda:0" if (torch.cuda.is_available()) else "cpu")
#v_48_010=version with 48 edges 0.10A noise
model_name = "v_48_020" #@param ["v_48_002", "v_48_010", "v_48_020", "v_48_030"]


backbone_noise=0.00               # Standard deviation of Gaussian noise to add to backbone atoms

path_to_model_weights='/content/ProteinMPNN/vanilla_model_weights'
hidden_dim = 128
num_layers = 3
model_folder_path = path_to_model_weights
if model_folder_path[-1] != '/':
    model_folder_path = model_folder_path + '/'
checkpoint_path = model_folder_path + f'{model_name}.pt'

checkpoint = torch.load(checkpoint_path, map_location=device)
print('Number of edges:', checkpoint['num_edges'])
noise_level_print = checkpoint['noise_level']
print(f'Training noise level: {noise_level_print}A')
model = ProteinMPNN(num_letters=21, node_features=hidden_dim, edge_features=hidden_dim, hidden_dim=hidden_dim, num_encoder_layers=num_layers, num_decoder_layers=num_layers, augment_eps=backbone_noise, k_neighbors=checkpoint['num_edges'])
model.to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Model loaded")

Number of edges: 48
Training noise level: 0.2A
Model loaded


In [ ]:
#@title Helper functions
def make_tied_positions_for_homomers(pdb_dict_list):
    my_dict = {}
    for result in pdb_dict_list:
        all_chain_list = sorted([item[-1:] for item in list(result) if item[:9]=='seq_chain']) #A, B, C, ...
        tied_positions_list = []
        chain_length = len(result[f"seq_chain_{all_chain_list[0]}"])
        for i in range(1,chain_length+1):
            temp_dict = {}
            for j, chain in enumerate(all_chain_list):
                temp_dict[chain] = [i] #needs to be a list
            tied_positions_list.append(temp_dict)
        my_dict[result['name']] = tied_positions_list
    return my_dict

# To ouput Score only, modify:

```
#@markdown ### Design Options
num_seqs = 1 #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
num_seq_per_target = num_seqs

#@markdown - Sampling temperature for amino acids, T=0.0 means taking argmax, T>>1.0 means sample randomly.
sampling_temp = "0.0001" #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5"]


save_score=1                      # 0 for False, 1 for True; save score=-log_prob to npy files
save_probs=1                      # 0 for False, 1 for True; save MPNN predicted probabilites per position
score_only=1                      # 0 for False, 1 for True; score input backbone-sequence pairs
conditional_probs_only=1          # 0 for False, 1 for True; output conditional probabilities p(s_i given the rest of the sequence and backbone)
```

In [ ]:
import re
from google.colab import files
import numpy as np

#########################
def get_pdb(pdb_code=""):
  if pdb_code is None or pdb_code == "":
    upload_dict = files.upload()
    pdb_string = upload_dict[list(upload_dict.keys())[0]]
    with open("tmp.pdb","wb") as out: out.write(pdb_string)
    return "tmp.pdb"
  else:
    os.system(f"wget -qnc https://files.rcsb.org/view/{pdb_code}.pdb")
    return f"{pdb_code}.pdb"

#@markdown ### Input Options
# pdb='1O91' #@param {type:"string"}
pdb='7CR5' #@param {type:"string"}
pdb_path = get_pdb(pdb)
#@markdown - pdb code (leave blank to get an upload prompt)

homomer = False #@param {type:"boolean"}
designed_chain = "L" #@param {type:"string"}
fixed_chain = "A H" #@param {type:"string"}

if designed_chain == "":
  designed_chain_list = []
else:
  designed_chain_list = re.sub("[^A-Za-z]+",",", designed_chain).split(",")

if fixed_chain == "":
  fixed_chain_list = []
else:
  fixed_chain_list = re.sub("[^A-Za-z]+",",", fixed_chain).split(",")

chain_list = list(set(designed_chain_list + fixed_chain_list))

#@markdown - specified which chain(s) to design and which chain(s) to keep fixed.
#@markdown   Use comma:`A,B` to specifiy more than one chain

#chain = "A" #@param {type:"string"}
#pdb_path_chains = chain
##@markdown - Define which chain to redesign

#@markdown ### Design Options
num_seqs = 1 #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
num_seq_per_target = num_seqs

#@markdown - Sampling temperature for amino acids, T=0.0 means taking argmax, T>>1.0 means sample randomly.
sampling_temp = "0.0001" #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5"]


save_score=1                      # 0 for False, 1 for True; save score=-log_prob to npy files
save_probs=1                      # 0 for False, 1 for True; save MPNN predicted probabilites per position
score_only=1                      # 0 for False, 1 for True; score input backbone-sequence pairs
conditional_probs_only=1          # 0 for False, 1 for True; output conditional probabilities p(s_i given the rest of the sequence and backbone)
conditional_probs_only_backbone=0 # 0 for False, 1 for True; if true output conditional probabilities p(s_i given backbone)

batch_size=1                      # Batch size; can set higher for titan, quadro GPUs, reduce this if running out of GPU memory
max_length=20000                  # Max sequence length

out_folder='.'                    # Path to a folder to output sequences, e.g. /home/out/
jsonl_path=''                     # Path to a folder with parsed pdb into jsonl
omit_AAs='X'                      # Specify which amino acids should be omitted in the generated sequence, e.g. 'AC' would omit alanine and cystine.

pssm_multi=0.0                    # A value between [0.0, 1.0], 0.0 means do not use pssm, 1.0 ignore MPNN predictions
pssm_threshold=0.0                # A value between -inf + inf to restric per position AAs
pssm_log_odds_flag=0               # 0 for False, 1 for True
pssm_bias_flag=0                   # 0 for False, 1 for True


##############################################################

folder_for_outputs = out_folder

NUM_BATCHES = num_seq_per_target//batch_size
BATCH_COPIES = batch_size
temperatures = [float(item) for item in sampling_temp.split()]
omit_AAs_list = omit_AAs
alphabet = 'ACDEFGHIKLMNPQRSTVWYX'

omit_AAs_np = np.array([AA in omit_AAs_list for AA in alphabet]).astype(np.float32)

chain_id_dict = None
fixed_positions_dict = None
pssm_dict = None
omit_AA_dict = None
bias_AA_dict = None
tied_positions_dict = None
bias_by_res_dict = None
bias_AAs_np = np.zeros(len(alphabet))


###############################################################
pdb_dict_list = parse_PDB(pdb_path, input_chain_list=chain_list)
dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=max_length)

chain_id_dict = {}
chain_id_dict[pdb_dict_list[0]['name']]= (designed_chain_list, fixed_chain_list)

print(chain_id_dict)
for chain in chain_list:
  l = len(pdb_dict_list[0][f"seq_chain_{chain}"])
  print(f"Length of chain {chain} is {l}")

if homomer:
  tied_positions_dict = make_tied_positions_for_homomers(pdb_dict_list)
else:
  tied_positions_dict = None

{'7CR5': (['L'], ['A', 'H'])}
Length of chain H is 222
Length of chain L is 213
Length of chain A is 125


In [ ]:
def _scores(S, log_probs, mask):
    """ Negative log probabilities """
    criterion = torch.nn.NLLLoss(reduction='none')
    loss = criterion(
        log_probs.contiguous().view(-1,log_probs.size(-1)),
        S.contiguous().view(-1)
    ).view(S.size())
    scores = torch.sum(loss * mask, dim=-1) / torch.sum(mask, dim=-1)
    return scores, loss

In [ ]:
#@title RUN
with torch.no_grad():
  print('Generating sequences...')
  for ix, protein in enumerate(dataset_valid):
    score_list = []
    loss_list = []
    all_probs_list = []
    all_log_probs_list = []
    S_sample_list = []
    batch_clones = [copy.deepcopy(protein) for i in range(BATCH_COPIES)]
    X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, visible_list_list, masked_list_list, masked_chain_length_list_list, chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, tied_pos_list_of_lists_list, pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, tied_beta = tied_featurize(batch_clones, device, chain_id_dict, fixed_positions_dict, omit_AA_dict, tied_positions_dict, pssm_dict, bias_by_res_dict)
    pssm_log_odds_mask = (pssm_log_odds_all > pssm_threshold).float() #1.0 for true, 0.0 for false
    name_ = batch_clones[0]['name']

    randn_1 = torch.randn(chain_M.shape, device=X.device)
    log_probs = model(X, S, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_1)
    mask_for_loss = mask*chain_M*chain_M_pos
    scores, losses = _scores(S, log_probs, mask_for_loss)
    native_score = scores.cpu().data.numpy()
    seq_loss = losses.cpu().data.numpy()

    for temp in temperatures:
        for j in range(NUM_BATCHES):
            randn_2 = torch.randn(chain_M.shape, device=X.device)
            if tied_positions_dict == None:
                sample_dict = model.sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), bias_by_res=bias_by_res_all)
                S_sample = sample_dict["S"]
            else:
                sample_dict = model.tied_sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), tied_pos=tied_pos_list_of_lists_list[0], tied_beta=tied_beta, bias_by_res=bias_by_res_all)
            # Compute scores
                S_sample = sample_dict["S"]
            log_probs = model(X, S_sample, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_2, use_input_decoding_order=True, decoding_order=sample_dict["decoding_order"])
            mask_for_loss = mask*chain_M*chain_M_pos
            scores, losses = _scores(S_sample, log_probs, mask_for_loss)
            print(scores)
            print(losses)
            scores = scores.cpu().data.numpy()
            seq_loss = losses.cpu().data.numpy()
            all_probs_list.append(sample_dict["probs"].cpu().data.numpy())
            all_log_probs_list.append(log_probs.cpu().data.numpy())
            S_sample_list.append(S_sample.cpu().data.numpy())
            for b_ix in range(BATCH_COPIES):
                masked_chain_length_list = masked_chain_length_list_list[b_ix]
                masked_list = masked_list_list[b_ix]
                seq_recovery_rate = torch.sum(torch.sum(torch.nn.functional.one_hot(S[b_ix], 21)*torch.nn.functional.one_hot(S_sample[b_ix], 21),axis=-1)*mask_for_loss[b_ix])/torch.sum(mask_for_loss[b_ix])
                seq = _S_to_seq(S_sample[b_ix], chain_M[b_ix])
                score = scores[b_ix]
                loss = seq_loss[b_ix]
                score_list.append(score)
                loss_list.append(loss)
                native_seq = _S_to_seq(S[b_ix], chain_M[b_ix])
                if b_ix == 0 and j==0 and temp==temperatures[0]:
                    start = 0
                    end = 0
                    list_of_AAs = []
                    for mask_l in masked_chain_length_list:
                        end += mask_l
                        list_of_AAs.append(native_seq[start:end])
                        start = end
                    native_seq = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
                    l0 = 0
                    for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
                        l0 += mc_length
                        native_seq = native_seq[:l0] + '/' + native_seq[l0:]
                        l0 += 1
                    sorted_masked_chain_letters = np.argsort(masked_list_list[0])
                    print_masked_chains = [masked_list_list[0][i] for i in sorted_masked_chain_letters]
                    sorted_visible_chain_letters = np.argsort(visible_list_list[0])
                    print_visible_chains = [visible_list_list[0][i] for i in sorted_visible_chain_letters]
                    native_score_print = np.format_float_positional(np.float32(native_score.mean()), unique=False, precision=4)
                    line = '>{}, score={}, fixed_chains={}, designed_chains={}, model_name={}\n{}\n'.format(name_, native_score_print, print_visible_chains, print_masked_chains, model_name, native_seq)
                    print(line.rstrip())
                start = 0
                end = 0
                list_of_AAs = []
                for mask_l in masked_chain_length_list:
                    end += mask_l
                    list_of_AAs.append(seq[start:end])
                    start = end

                seq = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
                l0 = 0
                for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
                    l0 += mc_length
                    seq = seq[:l0] + '/' + seq[l0:]
                    l0 += 1
                score_print = np.format_float_positional(np.float32(score), unique=False, precision=4)
                seq_rec_print = np.format_float_positional(np.float32(seq_recovery_rate.detach().cpu().numpy()), unique=False, precision=4)
                line = '>T={}, sample={}, score={}, seq_recovery={}\n{}\n'.format(temp,b_ix,score_print,seq_rec_print,seq)
                print(line.rstrip())


all_probs_concat = np.concatenate(all_probs_list)
all_log_probs_concat = np.concatenate(all_log_probs_list)
S_sample_concat = np.concatenate(S_sample_list)

Generating sequences...
tensor([0.6977])
tensor([[1.8384, 0.2010, 1.3230, 0.5531, 1.2759, 0.4278, 1.4663, 0.7848, 0.4804,
         0.3108, 0.3126, 0.3285, 1.0838, 1.2391, 0.1837, 0.5588, 0.3993, 0.3738,
         0.2368, 0.0863, 0.8243, 0.0782, 1.0567, 0.2817, 0.1008, 0.3041, 0.0822,
         0.9170, 1.1214, 1.5500, 1.2431, 0.5189, 0.2018, 0.6398, 0.0625, 0.6993,
         0.7286, 0.0775, 1.0129, 0.1136, 0.2818, 0.1746, 0.0971, 0.0908, 1.4633,
         1.0465, 0.2932, 0.5313, 1.4283, 0.3259, 1.0687, 1.0245, 0.7271, 0.0809,
         0.6022, 0.6899, 1.4091, 1.1517, 0.2656, 1.1687, 0.0939, 0.2565, 0.6579,
         0.9140, 0.1439, 0.6149, 0.5579, 0.0754, 0.2405, 0.5859, 0.3879, 0.1716,
         1.0445, 1.6046, 0.5616, 0.7556, 0.1103, 1.6113, 0.2287, 0.3711, 1.4561,
         0.1001, 1.1462, 1.5652, 1.2378, 0.1113, 1.1891, 0.0960, 0.7035, 0.0766,
         0.3263, 0.0721, 0.4472, 0.3313, 0.9663, 1.6994, 1.0726, 0.4824, 1.9746,
         0.9593, 0.4591, 0.4518, 0.0791, 0.9426, 0.0780, 0.0769, 1.8

In [ ]:
#@markdown ### Amino acid probabilties
import plotly.express as px
fig = px.imshow(np.exp(all_log_probs_concat).mean(0).T,
                labels=dict(x="positions", y="amino acids", color="probability"),
                y=list(alphabet),
                template="simple_white"
               )

fig.update_xaxes(side="top")


fig.show()

In [ ]:
#@markdown ### Sampling temperature adjusted amino acid probabilties
import plotly.express as px
fig = px.imshow(all_probs_concat.mean(0).T,
                labels=dict(x="positions", y="amino acids", color="probability"),
                y=list(alphabet),
                template="simple_white"
               )

fig.update_xaxes(side="top")


fig.show()

# Save Scores

NOTE: we need to pay attention to the LARGE position_scores, they are the one that needs to be changed.

In [ ]:
# Print and save position scores for each designed chain
for b_ix in range(BATCH_COPIES):
    # Get masked chain list and lengths
    masked_chain_length_list = masked_chain_length_list_list[b_ix]
    masked_list = masked_list_list[b_ix]

    # Process each designed chain
    for designed_chain in designed_chain_list:
        # Find index of designed chain
        chain_idx_list = [i for i, chain in enumerate(masked_list) if chain == designed_chain]

        if chain_idx_list:  # If chain is found
            chain_idx = chain_idx_list[0]  # Get first matching index

            # Calculate start and end positions in sequence
            start_idx = sum(masked_chain_length_list[:chain_idx])
            end_idx = start_idx + masked_chain_length_list[chain_idx]

            # Get scores for positions belonging to this chain
            chain_scores = seq_loss[b_ix, start_idx:end_idx]

            # Print scores
            print(f"Chain {designed_chain} position scores:")
            for pos, score in enumerate(chain_scores):
                if mask_for_loss[b_ix, start_idx + pos] > 0:  # Only print valid positions
                    print(f"Position {pos+1}: {score:.4f}")

            # Save scores to file (using chain ID in filename)
            chain_scores_output_file = os.path.join(out_folder, f"{name_}_chain_{designed_chain}_position_scores.npy")
            np.save(chain_scores_output_file, chain_scores)
            print(f"Chain {designed_chain} position scores saved to: {chain_scores_output_file}")

output_name = pdb_dict_list[0]['name'] # Get PDB name (e.g. '7CR5')

# Save probabilities
probs_output_file = os.path.join(out_folder, f"{output_name}_probs.npy")
np.save(probs_output_file, all_probs_concat)
print(f"Probabilities saved to: {probs_output_file}")

# Save log probabilities
log_probs_output_file = os.path.join(out_folder, f"{output_name}_log_probs.npy")
np.save(log_probs_output_file, all_log_probs_concat)
print(f"Log probabilities saved to: {log_probs_output_file}")

# Save scores
scores_output_file = os.path.join(out_folder, f"{output_name}_scores.npy")
np.save(scores_output_file, np.array(score_list))
print(f"Sample scores saved to: {scores_output_file}")

# Save native score
native_score_output_file = os.path.join(out_folder, f"{output_name}_native_score.npy")
np.save(native_score_output_file, native_score)
print(f"Native score saved to: {native_score_output_file}")

# Save total scores for each designed chain
for designed_chain in designed_chain_list:
    chain_scores_dict = {}
    for b_ix in range(BATCH_COPIES):
        masked_chain_length_list = masked_chain_length_list_list[b_ix]
        masked_list = masked_list_list[b_ix]

        chain_idx_list = [i for i, chain in enumerate(masked_list) if chain == designed_chain]
        if chain_idx_list:
            chain_idx = chain_idx_list[0]
            chain_scores_dict[f"batch_{b_ix}"] = {
                "scores": seq_loss[b_ix, sum(masked_chain_length_list[:chain_idx]):sum(masked_chain_length_list[:chain_idx]) + masked_chain_length_list[chain_idx]]
            }

    if chain_scores_dict:
        chain_scores_file = os.path.join(out_folder, f"{output_name}_chain_{designed_chain}_all_scores.npz")
        np.savez(chain_scores_file, **chain_scores_dict)
        print(f"Chain {designed_chain} all scores saved to: {chain_scores_file}")

Chain L position scores:
Position 1: 1.8384
Position 2: 0.2010
Position 3: 1.3230
Position 4: 0.5531
Position 5: 1.2759
Position 6: 0.4278
Position 7: 1.4663
Position 8: 0.7848
Position 9: 0.4804
Position 10: 0.3108
Position 11: 0.3126
Position 12: 0.3285
Position 13: 1.0838
Position 14: 1.2391
Position 15: 0.1837
Position 16: 0.5588
Position 17: 0.3993
Position 18: 0.3738
Position 19: 0.2368
Position 20: 0.0863
Position 21: 0.8243
Position 22: 0.0782
Position 23: 1.0567
Position 24: 0.2817
Position 25: 0.1008
Position 26: 0.3041
Position 27: 0.0822
Position 28: 0.9170
Position 29: 1.1214
Position 30: 1.5500
Position 31: 1.2431
Position 32: 0.5189
Position 33: 0.2018
Position 34: 0.6398
Position 35: 0.0625
Position 36: 0.6993
Position 37: 0.7286
Position 38: 0.0775
Position 39: 1.0129
Position 40: 0.1136
Position 41: 0.2818
Position 42: 0.1746
Position 43: 0.0971
Position 44: 0.0908
Position 45: 1.4633
Position 46: 1.0465
Position 47: 0.2932
Position 48: 0.5313
Position 49: 1.4283
Posi

In [ ]:
# import numpy as np
# import os

# file_path = "./7CR5_chain_L_position_scores.npy"
# if os.path.exists(file_path):
#     # Load the scores from the .npy file
#     position_scores = np.load(file_path)

#     # Print the loaded scores
#     print(f"Scores loaded from {file_path}:")
#     print(position_scores)

#     # Optionally, print position-wise scores more nicely
#     print("\nPosition-wise scores:")
#     for i, score in enumerate(position_scores):
#         print(f"Position {i+1}: {score:.4f}")
# else:
#     print(f"Error: File not found at {file_path}")

Scores loaded from ./7CR5_chain_L_position_scores.npy:
[1.8384038  0.20104067 1.3230045  0.55309325 1.2759491  0.42781866
 1.4663413  0.78484684 0.48040897 0.31080776 0.31258994 0.3284623
 1.0837721  1.2390715  0.18366964 0.5587697  0.39925867 0.37378785
 0.23683316 0.08625504 0.8243354  0.0781927  1.0566753  0.28166097
 0.10084082 0.3041024  0.08220071 0.9170292  1.1213641  1.5500087
 1.2430803  0.5188584  0.20179395 0.63975513 0.06251291 0.6993208
 0.72862595 0.07753224 1.012877   0.11356165 0.2818402  0.17456514
 0.0971478  0.09076081 1.4633485  1.0464834  0.293179   0.5312739
 1.4282852  0.3259244  1.068717   1.0244821  0.7271409  0.08091333
 0.6022087  0.68990064 1.4090598  1.1517096  0.26558384 1.1687487
 0.09388398 0.25646785 0.65791357 0.9140106  0.14391814 0.6149372
 0.5578586  0.07535698 0.24054506 0.5859015  0.38785118 0.17157988
 1.0444658  1.6045932  0.5616318  0.75556713 0.11031134 1.6112813
 0.22866645 0.3711041  1.4561425  0.10005182 1.1462266  1.5651852
 1.2377827  0.1

In [ ]:
print(all_probs_concat)

[[[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]]


In [ ]:
print(score_list)

[np.float32(0.69768995)]


In [ ]:
print(all_log_probs_concat)

[[[-3.2086728 -4.964147  -2.070362  ... -4.2362556 -4.407693  -5.3557134]
  [-5.9046984 -6.669267  -5.6345925 ... -5.686822  -5.23723   -5.71632  ]
  [-4.4596815 -6.1213026 -5.328101  ... -5.25144   -4.4276743 -5.5534277]
  ...
  [-3.383525  -5.2083035 -3.8571997 ... -5.014424  -4.316033  -5.479139 ]
  [-2.4328215 -5.668563  -4.979196  ... -5.2097297 -5.201787  -5.5953636]
  [-1.3700706 -4.998041  -3.2851439 ... -5.0871572 -4.9720006 -5.482556 ]]]


In [ ]:
print(native_score)

[1.2237136]
